In [ ]:
%cd ../..


In [ ]:
import pandas as pd
from scripts.features import *
from scripts.utils import *
from scripts.globals import RNADUPLEX_LOCATION, XGB_PIPELINE_DIR

pd.set_option('display.max_columns', None)


In [ ]:
clash_df = pd.read_csv("data/clash/clash_parsed.csv")
df = find_matches_with_rnaduplex(clash_df)


In [ ]:
# identifiers
df["id"] = clash_df["id"]
df["mirna_sequence"] = clash_df["mirna_sequence"]
df["mirna_accession"] = clash_df["mirna_accession"]
df["enst"] = clash_df["enst"]

# full seq
df["full_sequence_of_transcript"] = clash_df["full_sequence_of_transcript"]

# extended sequence columns
df["extended_mrna_sequence"] = clash_df["extended_sequence"]
df["extended_mrna_start"] = clash_df["extended_start"]
df["extended_mrna_end"] = clash_df["extended_end"]

# non-extended sequence columns
df["clash_mrna_sequence"] = clash_df["mrna_sequence"]
df["clash_mrna_start"] = clash_df["true_start_index"]
df["clash_mrna_end"] = clash_df["true_end_index"]

# true values
df["true_seed_type"] = clash_df["true_seed_type"]
df["true_num_basepairs"] = clash_df["num_basepairs"]
df["true_seed_basepairs"] = clash_df["seed_basepairs"]
df["true_energy"] = clash_df["folding_energy"]

# features

In [ ]:
df = generate_alignment_string_from_dot_bracket(df)
df = generate_match_count_columns(df)

# adding bp difference columns
df["bp_difference"] = df["pred_num_basepairs"] - df["true_num_basepairs"]
df["seed_bp_difference"] = df["pred_seed_basepairs"] - df["true_seed_basepairs"]

# getting mirna length
df["mirna_length"] = df["mirna_sequence"].str.len()

# using mirna length to figure out mre coordinates
df["mre_end"] = df["clash_mrna_start"] + df["mrna_end"] + df["mirna_start"]
df["mre_start"] = df["mre_end"] - df["mirna_length"]

# some start values are lower than zero, so we need to adjust
df['mre_start'] = df['mre_start'].clip(lower=0)

# creating mre sequence column
df['mre_region'] = df.apply(lambda row: row['full_sequence_of_transcript'][row['mre_start']:row['mre_end']], axis=1)


df.head()

In [ ]:
# writing to csv
df.to_csv(f"{XGB_PIPELINE_DIR}/1_find_matches_for_clash.csv", index=False)